In [35]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from tqdm import tqdm

In [5]:

# 实验数据（示例）
theta1 = torch.tensor([0.1, 0.2, 0.3, 0.4], dtype=torch.float32)  # 输入 theta1
theta2 = torch.tensor([0.5, 0.6, 0.7, 0.8], dtype=torch.float32)  # 输入 theta2
P_actual = torch.tensor([10.5, 12.3, 14.7, 18.2], dtype=torch.float32)  # 实验输出 P

# 定义力学模型
class MechanicsModel(nn.Module):
    def __init__(self):
        super(MechanicsModel, self).__init__()
        # 初始化待辨识参数 (k, b)
        self.k = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))  # 刚度参数
        self.b = nn.Parameter(torch.tensor(0.1, dtype=torch.float32))  # 阻尼参数

    def forward(self, theta1, theta2):
        # 力学模型公式 (示例)
        P_pred = self.k * theta1 + self.b * theta2
        return P_pred

# 实例化模型
model = MechanicsModel()

# 定义损失函数（均方误差）
loss_fn = nn.MSELoss()

# 定义优化器（Adam 优化器）
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 训练模型
num_epochs = 50000
for epoch in range(num_epochs):
    # 前向传播
    P_pred = model(theta1, theta2)
    loss = loss_fn(P_pred, P_actual)

    # 反向传播和优化
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 每隔 100 轮打印一次损失
    if (epoch + 1) % 10000 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# 输出辨识的参数
print(f"Estimated k: {model.k.item():.4f}")
print(f"Estimated b: {model.b.item():.4f}")

print(model.k * theta1 + model.b * theta2 - P_actual)

Epoch [10000/50000], Loss: 0.1870
Epoch [20000/50000], Loss: 0.1838
Epoch [30000/50000], Loss: 0.1838
Epoch [40000/50000], Loss: 0.1838
Epoch [50000/50000], Loss: 0.1838
Estimated k: 6.6250
Estimated b: 18.8750
tensor([-0.4000,  0.3500,  0.5000, -0.4500], grad_fn=<SubBackward0>)


In [7]:
data = pd.read_csv("../../log/realdata/StaticProcess/11group.csv")
print(data)
# 实验数据
P1_array = data['P1'].values
P2_array = data['P2'].values
theta1_array = data['theta1'].values
theta2_array = data['theta2'].values

      P1    P2      theta1     theta2
0    0.0   0.0   74.125730  60.829107
1    0.0   5.0   74.070626  60.930733
2    0.0  10.0   74.203975  44.451302
3    0.0  15.0   74.097930  33.965600
4    0.0  20.0   74.228239  33.742876
5    0.0  25.0   74.379173  17.582054
6    0.0  30.0   74.321364  17.637779
7    5.0   0.0   80.298920  54.034798
8    5.0   5.0   80.302943  54.526059
9    5.0  10.0   81.510336  33.695536
10   5.0  15.0   81.446567  24.472708
11   5.0  20.0   80.550773  25.503493
12   5.0  25.0   80.494328   9.421067
13   5.0  30.0   79.883559   7.701001
14  10.0   0.0   84.649965  49.946613
15  10.0   5.0   83.624100  51.243091
16  10.0  10.0   83.646988  29.855157
17  10.0  15.0   82.981763  28.536977
18  10.0  20.0   83.677227  14.726553
19  10.0  25.0   83.806022   5.942988
20  15.0   0.0   87.785691  46.342123
21  15.0   5.0   87.121623  47.739741
22  15.0  10.0   88.149620  25.679064
23  15.0  15.0   88.116583  15.346011
24  15.0  20.0   88.020780  15.398461
25  20.0   0

In [37]:
# 实验数据
theta1 = torch.tensor(theta1_array, dtype=torch.float32)*torch.pi/180  # 输入 theta1
theta2 = torch.tensor(theta2_array, dtype=torch.float32)*torch.pi/180  # 输入 theta2
P1_actual = torch.tensor(P1_array, dtype=torch.float32)  # 实验输出 P
P2_actual = torch.tensor(P2_array, dtype=torch.float32)  # 实验输出 P
P_actual = torch.stack([P1_actual, P2_actual], dim=1)
P_actual = P_actual*1000


# 定义力学模型
class MechanicsModel(nn.Module):
    def __init__(self):
        super(MechanicsModel, self).__init__()
        # 初始化待辨识参数：k_3, k_4, m_1, m_2, m_3, m_4, l_10, l_20, S_1, S_2
        self.k_3 = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))
        self.k_4 = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))
        self.m_1 = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))
        self.m_2 = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))
        self.m_3 = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))
        self.m_4 = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))
        self.l_10 = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))
        self.l_20 = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))
        self.S_1 = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))
        self.S_2 = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))

        
    def forward(self, theta_1, theta_2):
        # 静力学模型公式        
        # 固定参数
        a_1, a_2, b_1, b_2, d_1, d_2 = 0.25, 0.25, 0.21213, 0.1, 0.06, 0.10
        beta_1, beta_2 = 8.13 / 180 * torch.pi, 30 / 180 * torch.pi
        g = 9.8
        # 优化参数
        # k_3, k_4, m_1, m_2, m_3, m_4, l_10, l_20, S_1, S_2 = params

        A_x_O = d_1
        A_y_O = 0

        B_x_O = -d_2
        B_y_O = 0

        C_x_O = b_1 * torch.cos(theta_1 - beta_1)
        C_y_O = b_1 * torch.sin(theta_1 - beta_1)

        D_x_O = a_1 * torch.cos(theta_1) + b_2 * torch.cos(theta_1 + theta_2 + beta_2)
        D_y_O = a_1 * torch.sin(theta_1) + b_2 * torch.sin(theta_1 + theta_2 + beta_2)

        E_x_O = a_1 * torch.cos(theta_1)
        E_y_O = a_1 * torch.sin(theta_1)

        F_x_O = a_1 * torch.cos(theta_1) + a_2 * torch.cos(theta_1 + theta_2)
        F_y_O = a_1 * torch.sin(theta_1) + a_2 * torch.sin(theta_1 + theta_2)

        # 计算偏导数
        # 对 theta_1 的偏导数

        dA_x_O_dtheta_1 = 0
        dA_y_O_dtheta_1 = 0

        dB_x_O_dtheta_1 = 0
        dB_y_O_dtheta_1 = 0

        dC_x_O_dtheta_1 = -b_1 * torch.sin(theta_1 - beta_1)
        dC_y_O_dtheta_1 = b_1 * torch.cos(theta_1 - beta_1)

        dD_x_O_dtheta_1 = -a_1 * torch.sin(theta_1) - b_2 * torch.sin(theta_1 + theta_2 + beta_2)
        dD_y_O_dtheta_1 = a_1 * torch.cos(theta_1) + b_2 * torch.cos(theta_1 + theta_2 + beta_2)

        dE_x_O_dtheta_1 = -a_1 * torch.sin(theta_1)
        dE_y_O_dtheta_1 = a_1 * torch.cos(theta_1)

        dF_x_O_dtheta_1 = -a_1 * torch.sin(theta_1) - a_2 * torch.sin(theta_1 + theta_2)
        dF_y_O_dtheta_1 = a_1 * torch.cos(theta_1) + a_2 * torch.cos(theta_1 + theta_2)

        # 对 theta_2 的偏导数

        dA_x_O_dtheta_2 = 0
        dA_y_O_dtheta_2 = 0

        dB_x_O_dtheta_2 = 0
        dB_y_O_dtheta_2 = 0

        dC_x_O_dtheta_2 = 0
        dC_y_O_dtheta_2 = 0

        dD_x_O_dtheta_2 = -b_2 * torch.sin(theta_1 + theta_2 + beta_2)
        dD_y_O_dtheta_2 = b_2 * torch.cos(theta_1 + theta_2 + beta_2)

        dE_x_O_dtheta_2 = 0
        dE_y_O_dtheta_2 = 0

        dF_x_O_dtheta_2 = -a_2 * torch.sin(theta_1 + theta_2)
        dF_y_O_dtheta_2 = a_2 * torch.cos(theta_1 + theta_2)

        # 计算长度 l1 和 l2
        l_1 = torch.sqrt((A_x_O - C_x_O)**2 + (A_y_O - C_y_O)**2)
        l_2 = torch.sqrt((B_x_O - D_x_O)**2 + (B_y_O - D_y_O)**2)

        # 计算偏导数
        # 偏导数 d/dtheta_1
        dl_1_dtheta_1 = 1/l_1 * ((A_x_O - C_x_O) * (dA_x_O_dtheta_1 - dC_x_O_dtheta_1) + (A_y_O - C_y_O) * (dA_y_O_dtheta_1 - dC_y_O_dtheta_1))
        dl_2_dtheta_1 = 1/l_2 * ((B_x_O - D_x_O) * (dB_x_O_dtheta_1 - dD_x_O_dtheta_1) + (B_y_O - D_y_O) * (dB_y_O_dtheta_1 - dD_y_O_dtheta_1))

        # 偏导数 d/dtheta_2
        dl_1_dtheta_2 = 1/l_1 * ((A_x_O - C_x_O) * (dA_x_O_dtheta_2 - dC_x_O_dtheta_2) + (A_y_O - C_y_O) * (dA_y_O_dtheta_2 - dC_y_O_dtheta_2))
        dl_2_dtheta_2 = 1/l_2 * ((B_x_O - D_x_O) * (dB_x_O_dtheta_2 - dD_x_O_dtheta_2) + (B_y_O - D_y_O) * (dB_y_O_dtheta_2 - dD_y_O_dtheta_2))

        # print(dl_1_dtheta_2)  # check the model

        # 等式右侧
        RHSb_1 = -(self.m_1*g*(dE_y_O_dtheta_1/2) + self.m_2*g*((dE_y_O_dtheta_1+dF_y_O_dtheta_1)/2) + self.m_3*g*(dC_y_O_dtheta_1/2) + self.m_4*g*(dD_y_O_dtheta_1/2) )
        RHSb_2 = -(self.m_1*g*(dE_y_O_dtheta_2/2) + self.m_2*g*((dE_y_O_dtheta_2+dF_y_O_dtheta_2)/2) + self.m_3*g*(dC_y_O_dtheta_2/2) + self.m_4*g*(dD_y_O_dtheta_2/2) )
        b = torch.stack([RHSb_1, RHSb_2], dim=1)      # 注意，不能使用tensor创建，GPT推荐使用torch.stack
        # print(b.shape)

        # 等式左侧
        LHSA = torch.stack([
            torch.stack([dl_1_dtheta_1, dl_2_dtheta_1], dim=1), 
            torch.stack([dl_1_dtheta_2, dl_2_dtheta_2], dim=1)
            ], dim=1)

        # 回复力
        F_k = torch.stack([self.k_3*(l_1-self.l_10), self.k_4*(l_2-self.l_20)], dim=1)

        # 计算静力学
        StaticForce = torch.linalg.solve(LHSA, b) + F_k
        StaticP_pred = torch.stack([StaticForce[:, 0]/self.S_1, StaticForce[:, 1]/self.S_2], dim=1)
        # print(StaticP_pred)
        return StaticP_pred


# 实例化模型
model = MechanicsModel()

# 定义损失函数（均方误差）
loss_fn = nn.MSELoss()

# 定义优化器（Adam 优化器）
optimizer = optim.Adam(model.parameters(), lr=0.1)

# 训练模型
num_epochs = 500000
for epoch in tqdm(range(num_epochs)):
    # 前向传播
    P_pred = model(theta1, theta2)
    loss = loss_fn(P_pred, P_actual)

    # 反向传播和优化
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 每隔 100 轮打印一次损失
    if (epoch + 1) % 10000 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# # 输出辨识的参数
# print(f"Estimated k: {model.k.item():.4f}")
# print(f"Estimated b: {model.b.item():.4f}")

# print(model.k * theta1 + model.b * theta2 - P_actual)

  2%|▏         | 10191/500000 [00:07<05:42, 1429.41it/s]

Epoch [10000/500000], Loss: 3961873.0000


  4%|▍         | 20281/500000 [00:14<05:36, 1426.62it/s]

Epoch [20000/500000], Loss: 3878060.7500


  6%|▌         | 30202/500000 [00:21<05:29, 1425.80it/s]

...

Epoch [480000/500000], Loss: 3639764.0000


 98%|█████████▊| 490218/500000 [06:09<00:07, 1326.28it/s]

Epoch [490000/500000], Loss: 3632042.7500


100%|██████████| 500000/500000 [06:17<00:00, 1324.67it/s]

Epoch [500000/500000], Loss: 3649395.2500


In [38]:
# 输出所有参数
print(f"Estimated k3: {model.k_3.item():.4f}")
print(f"Estimated k4: {model.k_4.item():.4f}")
print(f"Estimated m1: {model.m_1.item():.4f}")
print(f"Estimated m2: {model.m_2.item():.4f}")
print(f"Estimated m3: {model.m_3.item():.4f}")
print(f"Estimated m4: {model.m_4.item():.4f}")


Estimated k3: 145.5782
Estimated k4: 86.1714
Estimated m1: 1384.7881
Estimated m2: 837.5090
Estimated m3: 1065.2268
Estimated m4: -460.4176
